In [1]:
import sys

print(sys.executable)
print(sys.version)

/Users/laplace/Documents/work/ecommerce-growth-analytics/.venv/bin/python
3.14.6 (v3.14.6:c63aec69bd5, Jun 10 2026, 08:07:54) [Clang 21.0.0 (clang-2100.1.1.101)]


# E-Commerce Growth Analytics

## Phase 1 — Source Data Audit

This notebook examines the structure, quality, relationships, and business meaning of the original Olist e-commerce datasets before transformation or synthetic extension.

In [2]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
DATA_PATH = Path("../data/source/olist")

print("Data path:", DATA_PATH.resolve())
print("Folder exists:", DATA_PATH.exists())

Data path: /Users/laplace/Documents/work/ecommerce-growth-analytics/data/source/olist
Folder exists: True


In [4]:
csv_files = sorted(DATA_PATH.glob("*.csv"))

print(f"Number of CSV files found: {len(csv_files)}")

for file in csv_files:
    print(file.name)

Number of CSV files found: 9
olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
datasets = {}

for file in csv_files:
    datasets[file.stem] = pd.read_csv(file)

print(f"Successfully loaded {len(datasets)} datasets.")

Successfully loaded 9 datasets.


In [6]:
for dataset_name in datasets:
    print(dataset_name)

olist_customers_dataset
olist_geolocation_dataset
olist_order_items_dataset
olist_order_payments_dataset
olist_order_reviews_dataset
olist_orders_dataset
olist_products_dataset
olist_sellers_dataset
product_category_name_translation


In [7]:
inventory_records = []

for dataset_name, df in datasets.items():
    inventory_records.append({
        "dataset": dataset_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_cells": int(df.isna().sum().sum()),
        "memory_mb": round(df.memory_usage(deep=True).sum() / 1_048_576, 2)
    })

dataset_inventory = (
    pd.DataFrame(inventory_records)
    .sort_values("rows", ascending=False)
    .reset_index(drop=True)
)

dataset_inventory

,dataset,rows,columns,duplicate_rows,missing_cells,memory_mb
0,olist_geolocation_dataset,1000163,5,261831,0,130.26
1,olist_order_items_dataset,112650,7,0,0,35.99
2,olist_order_payments_dataset,103886,5,0,0,16.23
3,olist_customers_dataset,99441,5,0,0,26.59
4,olist_orders_dataset,99441,8,0,4908,52.94
5,olist_order_reviews_dataset,99224,7,0,145903,39.12
6,olist_products_dataset,32951,9,0,2448,6.30
7,olist_sellers_dataset,3095,4,0,0,0.59
8,product_category_name_translation,71,2,0,0,0.01


In [8]:
for dataset_name, df in datasets.items():
    print("=" * 100)
    print(f"DATASET: {dataset_name}")
    print(f"SHAPE: {df.shape}")
    display(df.head(3))

DATASET: olist_customers_dataset
SHAPE: (99441, 5)


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


DATASET: olist_geolocation_dataset
SHAPE: (1000163, 5)


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP


DATASET: olist_order_items_dataset
SHAPE: (112650, 7)


...

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00


DATASET: olist_sellers_dataset
SHAPE: (3095, 4)


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ


DATASET: product_category_name_translation
SHAPE: (71, 2)


,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto


# Dataset Catalogue

The Olist database contains nine related datasets. Each table represents a different business process within the e-commerce platform.

The objective of this section is to understand the purpose, grain, and relationships of each dataset before performing any analysis.

In [9]:
def inspect_dataset(name, df):
    print("=" * 100)
    print(f"DATASET: {name}")
    print("=" * 100)

    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")

    print("\nColumn Names")
    print("-" * 50)

    for column in df.columns:
        print(column)

    print("\nData Types")
    print("-" * 50)

    display(df.dtypes.to_frame("Data Type"))

    print("\nFirst Five Records")
    print("-" * 50)

    display(df.head())

    print("\nMissing Values")
    print("-" * 50)

    display(df.isna().sum().to_frame("Missing Values"))

In [10]:
inspect_dataset(
    "Customers",
    datasets["olist_customers_dataset"]
)

DATASET: Customers
Rows: 99,441
Columns: 5

Column Names
--------------------------------------------------
customer_id
customer_unique_id
customer_zip_code_prefix
customer_city
customer_state

Data Types
--------------------------------------------------


,Data Type
customer_id,str
customer_unique_id,str
customer_zip_code_prefix,int64
customer_city,str
customer_state,str



First Five Records
--------------------------------------------------


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP



Missing Values
--------------------------------------------------


,Missing Values
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


In [11]:
inspect_dataset(
    "Orders",
    datasets["olist_orders_dataset"]
)

DATASET: Orders
Rows: 99,441
Columns: 8

Column Names
--------------------------------------------------
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date

Data Types
--------------------------------------------------


,Data Type
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,str
order_approved_at,str
order_delivered_carrier_date,str
order_delivered_customer_date,str
order_estimated_delivery_date,str



First Five Records
--------------------------------------------------


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Missing Values
--------------------------------------------------


,Missing Values
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


In [12]:
inspect_dataset(
    "Orders",
    datasets["olist_orders_dataset"]
)

DATASET: Orders
Rows: 99,441
Columns: 8

Column Names
--------------------------------------------------
order_id
customer_id
order_status
order_purchase_timestamp
order_approved_at
order_delivered_carrier_date
order_delivered_customer_date
order_estimated_delivery_date

Data Types
--------------------------------------------------


,Data Type
order_id,str
customer_id,str
order_status,str
order_purchase_timestamp,str
order_approved_at,str
order_delivered_carrier_date,str
order_delivered_customer_date,str
order_estimated_delivery_date,str



First Five Records
--------------------------------------------------


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00



Missing Values
--------------------------------------------------


,Missing Values
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


## Analytical Questions – Orders Table

During the initial audit, the following business questions were identified for further investigation:

1. Why are some values in `order_delivered_customer_date` missing?
2. Are missing delivery timestamps associated with specific `order_status` values?
3. Are there any orders marked as **delivered** that do not have a delivery timestamp?
4. Why are some values in `order_approved_at` missing?
5. Are there orders that were purchased but never approved?
6. Does every approved order eventually reach the customer?

## Initial Observations – Orders Table

### Observation 1

The Orders table serves as the central transactional table within the Olist database. It connects customers to the remaining operational datasets.

### Observation 2

The table contains five timestamp columns that describe different stages of the order lifecycle, from purchase through estimated delivery.

### Observation 3

Missing values are concentrated in the operational timestamp fields rather than customer or order identifiers, suggesting that they may represent incomplete order processes rather than poor data quality.

### Observation 4

Further investigation is required to determine whether missing delivery timestamps correspond to cancelled, unavailable, or undelivered orders.

In [13]:
inspect_dataset(
    "Order Items",
    datasets["olist_order_items_dataset"]
)

DATASET: Order Items
Rows: 112,650
Columns: 7

Column Names
--------------------------------------------------
order_id
order_item_id
product_id
seller_id
shipping_limit_date
price
freight_value

Data Types
--------------------------------------------------


,Data Type
order_id,str
order_item_id,int64
product_id,str
seller_id,str
shipping_limit_date,str
price,float64
freight_value,float64



First Five Records
--------------------------------------------------


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14



Missing Values
--------------------------------------------------


,Missing Values
order_id,0
order_item_id,0
product_id,0
seller_id,0
shipping_limit_date,0
price,0
freight_value,0


In [14]:
datasets["olist_order_items_dataset"]["order_id"].value_counts().head(10)

order_id
8272b63d03f5f79c56e9e4120aec44ef    21
1b15974a0141d54e36626dca3fdc731a    20
ab14fdcfbe524636d65ee38360e22ce8    20
428a2f660dc84138d969ccd69a0ab6d5    15
9ef13efd6949e4573a18964dd1bbe7f5    15
73c8ab38f07dc94389065f7eba4f297a    14
9bdc4d4c71aa1de4606060929dee888c    14
37ee401157a3a0b28c9c6d0ed8c3b24b    13
2c2a19b5703863c908512d135aa6accc    12
3a213fcdfe7d98be74ea0dc05a8b31ae    12
Name: count, dtype: int64

In [15]:
datasets["olist_orders_dataset"]["order_id"].value_counts().head(10)


order_id
e481f51cbdc54678b7cc49136f2d6af7    1
53cdb2fc8bc7dce0b6741e2150273451    1
47770eb9100c2d0c44946d9cf07ec65d    1
949d5b44dbf5de918fe9c16f97b45f8a    1
ad21c59c0840e6cb83a9ceb5573f8159    1
a4591c265e18cb1dcee52889e2d8acc3    1
136cce7faa42fdb2cefd53fdc79a6098    1
6514b8ad8028c9f2cc2374ded245783f    1
76c6e866289321a7c93b82b54852dc33    1
e69bfb5eb88e0ed6a785585b27e16dbf    1
Name: count, dtype: int64

In [16]:
datasets["olist_order_items_dataset"]["order_item_id"].value_counts().sort_index()

order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64

In [17]:
inspect_dataset(
    "Products",
    datasets["olist_products_dataset"]
)

DATASET: Products
Rows: 32,951
Columns: 9

Column Names
--------------------------------------------------
product_id
product_category_name
product_name_lenght
product_description_lenght
product_photos_qty
product_weight_g
product_length_cm
product_height_cm
product_width_cm

Data Types
--------------------------------------------------


,Data Type
product_id,str
product_category_name,str
product_name_lenght,float64
product_description_lenght,float64
product_photos_qty,float64
product_weight_g,float64
product_length_cm,float64
product_height_cm,float64
product_width_cm,float64



First Five Records
--------------------------------------------------


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00



Missing Values
--------------------------------------------------


,Missing Values
product_id,0
product_category_name,610
product_name_lenght,610
product_description_lenght,610
product_photos_qty,610
product_weight_g,2
product_length_cm,2
product_height_cm,2
product_width_cm,2


In [ ]:
datasets["olist_products_dataset"]["product_id"].value_counts().sort_index(ascending=True )

product_id
00066f42aeeb9f3007548bb9d3f33c38    1
00088930e925c41fd95ebfe695fd2655    1
0009406fd7479715e4bef61dd91f2462    1
000b8f95fcb9e0096488278317764d19    1
000d9be29b5207b54e86aa1b1ac54872    1
                                   ..
fff6177642830a9a94a0f2cba5e476d1    1
fff81cc3158d2725c0655ab9ba0f712c    1
fff9553ac224cec9d15d49f5a263411f    1
fffdb2d0ec8d6a61f0a0a0db3f25b441    1
fffe9eeff12fcbd74a2f2b007dde0c58    1
Name: count, Length: 32951, dtype: int64

In [23]:
inspect_dataset(
    "Sellers",
    datasets["olist_sellers_dataset"]
)

DATASET: Sellers
Rows: 3,095
Columns: 4

Column Names
--------------------------------------------------
seller_id
seller_zip_code_prefix
seller_city
seller_state

Data Types
--------------------------------------------------


,Data Type
seller_id,str
seller_zip_code_prefix,int64
seller_city,str
seller_state,str



First Five Records
--------------------------------------------------


,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP



Missing Values
--------------------------------------------------


,Missing Values
seller_id,0
seller_zip_code_prefix,0
seller_city,0
seller_state,0


In [25]:
inspect_dataset(
    "Order Payments",
    datasets["olist_order_payments_dataset"])

DATASET: Order Payments
Rows: 103,886
Columns: 5

Column Names
--------------------------------------------------
order_id
payment_sequential
payment_type
payment_installments
payment_value

Data Types
--------------------------------------------------


,Data Type
order_id,str
payment_sequential,int64
payment_type,str
payment_installments,int64
payment_value,float64



First Five Records
--------------------------------------------------


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45



Missing Values
--------------------------------------------------


,Missing Values
order_id,0
payment_sequential,0
payment_type,0
payment_installments,0
payment_value,0
